# einops-repeat composite — cx6: broadcast a triangle across NR rays via einops.repeat, then evaluate rays at u

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `ray-parametric-form`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-repeat"
DD_ATOM_IDS = ["einops-repeat", "ray-parametric-form"]
DD_SUBTOPICS = ["Einops: Repeat", "Geometry: Ray parametric form"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

When you intersect `NR` rays against a SINGLE triangle, the triangle vertices `(A, B, C)` have shape `(3, 3)` — but to align them with `(NR, 3)` ray data they need to be repeated across the ray axis. `einops.repeat(triangle, 'v d -> nr v d', nr=NR)` is the explicit broadcast — a stride-0 view, no copy, with the named axis documenting WHICH dim is broadcasting.

Combined with `ray-parametric-form` `R(u) = O + u*D`, this composition lets you compute, for a batch of rays and a batch of parameter values, the 3D points along each ray AND have the triangle data pre-aligned for the downstream intersection test — all in pure tensor ops, no Python loops over rays.

The point: `einops.repeat` is the explicit-broadcast tool — it's what you reach for when implicit broadcasting would be ambiguous (which axis broadcasts where?). Here the triangle needs to be aligned against the ray axis, and the named pattern makes that explicit.

### Composite Exercise — broadcast a triangle across NR rays via einops.repeat, then evaluate rays at u

**Atoms exercised together**: `einops-repeat`, `ray-parametric-form`

Implement `cx6_eval_rays_with_triangle(rays, us, triangle)` that:

1. Evaluates each ray at its corresponding parameter (`ray-parametric-form`).
2. Broadcasts the single triangle across all rays (`einops.repeat`).

Inputs:
- `rays`: `(NR, 2, 3)` — `rays[r, 0]` is origin, `rays[r, 1]` is direction.
- `us`: `(NR,)` — one parameter value per ray.
- `triangle`: `(3, 3)` — three vertices of a single triangle, rows = vertices.

Returns `(points, tri_broadcast)`:
- `points: (NR, 3)` — `O_r + us[r] * D_r` (the parametric form, one point per ray).
- `tri_broadcast: (NR, 3, 3)` — the same triangle, broadcast across `NR` rays via `repeat(triangle, 'v d -> nr v d', nr=NR)`.

The test asserts that `tri_broadcast` is a true zero-copy view (shares storage with `triangle`) and that `points` matches the manual parametric evaluation.

In [ ]:
def cx6_eval_rays_with_triangle(rays, us, triangle):
    NR = rays.shape[0]
    O = rays[:, 0]                  # (NR, 3)
    D = rays[:, 1]                  # (NR, 3)
    # ray-parametric-form: R(u) = O + u * D, with us broadcast over the 3-axis.
    points = O + us.unsqueeze(-1) * D
    # einops-repeat: explicit broadcast of the single triangle across the NR axis.
    # Stride-0 view — no copy, storage shared with `triangle`.
    tri_broadcast = repeat(triangle, 'v d -> nr v d', nr=NR)
    return points, tri_broadcast


<details><summary>Show solution — cx6</summary>

```python
def cx6_eval_rays_with_triangle(rays, us, triangle):
    NR = rays.shape[0]
    O = rays[:, 0]                  # (NR, 3)
    D = rays[:, 1]                  # (NR, 3)
    # ray-parametric-form: R(u) = O + u * D, with us broadcast over the 3-axis.
    points = O + us.unsqueeze(-1) * D
    # einops-repeat: explicit broadcast of the single triangle across the NR axis.
    # Stride-0 view — no copy, storage shared with `triangle`.
    tri_broadcast = repeat(triangle, 'v d -> nr v d', nr=NR)
    return points, tri_broadcast
```

Two cheap operations, both load-bearing. `us.unsqueeze(-1) * D` is the textbook broadcast for the parametric form — it lines up `(NR, 1)` against `(NR, 3)` so each ray's scalar parameter scales its own 3-vector direction. `repeat(triangle, 'v d -> nr v d', nr=NR)` is the explicit version of `triangle.expand(NR, 3, 3)` — same stride-0 storage, but the named pattern makes intent obvious to the next reader.

This composition is the per-ray data prep step right before a ray-triangle intersection: rays evaluated, triangle broadcast — then downstream code can `stack([-D, B-A, C-A], dim=-1)` to build the `(NR, 3, 3)` LHS matrix from cx1, and pass it to the batched solve from cx3.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["Einops: Repeat", "Geometry: Ray parametric form"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()